In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [3]:
data=pd.read_csv('Churn_Modelling.csv')

In [4]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [5]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])


In [6]:
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [7]:
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

In [8]:
X = data.drop('Exited', axis=1)
y = data['Exited']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [ ]:
#adding hidden layers for hyperparametertunning

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model

In [14]:
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [15]:
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [17]:
grid= GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Epoch 1/50


c:\Users\Athul VR\OneDrive\Desktop\Churn modeling using ANN\myvenv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\Athul VR\OneDrive\Desktop\Churn modeling using ANN\myvenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 798us/step - accuracy: 0.7946 - loss: 0.4648
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 706us/step - accuracy: 0.8294 - loss: 0.4047
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 692us/step - accuracy: 0.8430 - loss: 0.3822
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 696us/step - accuracy: 0.8493 - loss: 0.3652
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 700us/step - accuracy: 0.8530 - loss: 0.3568
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 836us/step - accuracy: 0.8571 - loss: 0.3508
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 700us/step - accuracy: 0.8564 - loss: 0.3466
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 728us/step - accuracy: 0.8587 - loss: 0.3446
Epoch 9/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 705us/step - accuracy: 0.8586 - loss: 0.3420
Epoch 10/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 711us/step - accuracy: 0.8600 - loss: 0.3410
Epoch 11/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 739us/step - accuracy: 0.8590 - loss: 0.3398
Epoch 12/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 